In [1]:
!pip -q install pythainlp
!pip install spacy_thai
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.3/19.3 MB 80.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.7/15.7 MB 59.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 953.8/953.8 kB 39.0 MB/s eta 0:00:00


In [2]:

import re
import unicodedata
import nltk
import random
import numpy as np
import pandas as pd
from pythainlp.tokenize import word_tokenize
from pythainlp import word_vector
from pythainlp.corpus.common import thai_stopwords
# import seaborn as sns
# import matplotlib.pyplot as plt

# try:
#     nltk.data.find('tokenizers/punkt')
# except LookupError:
#     nltk.download('punkt')

# try:
#     nltk.data.find('corpora/stopwords')
# except LookupError:
#     nltk.download('stopwords')
import torch.nn as nn
from sklearn.metrics import f1_score
from gensim.models import Word2Vec
from collections import Counter
# Setting as large the xtick and ytick font sizes in graphs

# plt.rcParams['xtick.labelsize'] = 'large'
# plt.rcParams['ytick.labelsize'] = 'large'

/usr/local/lib/python3.12/dist-packages/google/cloud/aiplatform/models.py:52: FutureWarning: Support for google-cloud-storage < 3.0.0 will be removed in a future version of google-cloud-aiplatform. Please upgrade to google-cloud-storage >= 3.0.0.
  from google.cloud.aiplatform.utils import gcs_utils


In [3]:
LABEL_COLS = [
    "politics", "human_rights", "quality_of_life", "international",
    "social", "environment", "economics", "culture", "labor",
    "national_security", "ict", "education"
]

RANDOM_STATE= 42
BATCH_SIZE = 16
LR = 2e-4
EPOCHS = 5
THRESHOLD=0.5

# MLP Class Config
HIDDEN_DIM=300
MAX_LEN = 10
NUM_LABEL = len(LABEL_COLS)

MAX_VOCAB=100
PAD_TOKEN = "<pad>"
UNK_TOKEN = "<unk>"
PAD_ID = 0
UNK_ID = 1

# Prachatai dataset

In [4]:
# Storing the csv file into a DataFrame "df"

df_train = pd.read_csv('/kaggle/input/datasets/ratthachat/pythainlp-prachatai-67k/prachatai_train.csv').head(400)
df_valid = pd.read_csv('/kaggle/input/datasets/ratthachat/pythainlp-prachatai-67k/prachatai_validation.csv').head(200)
df_test = pd.read_csv('/kaggle/input/datasets/ratthachat/pythainlp-prachatai-67k/prachatai_test.csv').head(200)
print("df_train:", df_train.shape)
print("df_valid:", df_valid.shape)
print("df_test:", df_test.shape)

df_train: (400, 17)
df_valid: (200, 17)
df_test: (200, 17)


In [5]:
X_train = df_train['body_text']
y_train = df_train[LABEL_COLS].astype(int)

X_val = df_valid['body_text']
y_val = df_valid[LABEL_COLS].astype(int)

X_test = df_test['body_text']
y_test = df_test[LABEL_COLS].astype(int)

X_test.tail(3),y_test.tail(3)

(197    สิทธิพลเมือง สัญชาติไทยใครกำหนด\n\nคดีแม่อายเป...
 198    ฉันเจอวลีที่บอกว่า “ผู้หญิงอย่าหยุดสวย” ครั้งแ...
 199    สัมภาษณ์โดย ณภัค เสรีรักษ์\n\n \n\n \n\n \n\nใ...
 Name: body_text, dtype: object,
      politics  human_rights  quality_of_life  international  social  \
 197         1             0                0              0       0   
 198         0             0                1              0       1   
 199         0             0                0              0       0   
 
      environment  economics  culture  labor  national_security  ict  education  
 197            0          0        0      0                  0    0          0  
 198            0          0        1      0                  0    0          0  
 199            0          0        0      0                  0    0          0  )

In [6]:
# แปลงเลขไทยเป็นเลขอารบิก (๑๒๓ และ ๐๑๒๓)
_THAI_DIGITS = str.maketrans("๐๑๒๓๔๕๖๗๘๙๑๒๓๔๕๖๗๘๙", "0123456789123456789")
# หมายเหตุ: ชุดนี้ครอบทั้งเลขไทย ๐-๙ และเลขไทยแบบ ๑-๙ (ใช้กันในเอกสาร)

def process(
    x: str,
    *,
    lower: bool = True,
    keep_punct: bool = False,   # True ถ้าอยากเก็บ !?.,:;() ไว้
    keep_newlines: bool = False # True ถ้าอยากคง \n เป็นตัวแบ่งย่อหน้า
):
    # 1) normalize unicode (ช่วยเรื่องตัวอักษร/ช่องว่างแปลกๆ)
    x = unicodedata.normalize("NFKC", str(x))

    # 2) จัดการ NBSP / zero-width / BOM ที่เจอบ่อยในข่าว-บทความ
    x = x.replace("\u00A0", " ")   # NBSP = \xa0
    x = x.replace("\ufeff", "")    # BOM
    x = re.sub(r"[\u200b-\u200f\u202a-\u202e]", "", x)  # zero-width + bidi marks

    # 3) ลบ HTML tags / entities ง่ายๆ
    x = re.sub(r"<[^>]+>", " ", x)
    x = x.replace("&nbsp;", " ")

    # 4) ลบ URL / อีเมล (ถ้าต้องการ)
    x = re.sub(r"https?://\S+|www\.\S+", " ", x)
    x = re.sub(r"\b[\w\.-]+@[\w\.-]+\.\w+\b", " ", x)

    # 5) แปลงเลขไทยเป็นเลขอารบิก
    x = x.translate(_THAI_DIGITS)

    # 6) เก็บเฉพาะตัวอักษรที่ต้องการ: ไทย/อังกฤษ/ตัวเลข + ช่องว่าง + (เลือก) punctuation
    if keep_punct:
        # เก็บ . , ! ? : ; ( ) " ' - รวมถึง ๆ ฯ ฯลฯ (เพิ่มได้เอง)
        x = re.sub(r"[^0-9A-Za-zก-๙\s\.\,\!\?\:\;\(\)\"\'\-ๆฯ]", " ", x)
    else:
        x = re.sub(r"[^0-9A-Za-zก-๙\s]", " ", x)

    # 7) จัดการ newline / whitespace
    if keep_newlines:
        # normalize ช่องว่างในแต่ละบรรทัด แต่คง \n ไว้
        x = re.sub(r"[ \t]+", " ", x)
        x = re.sub(r"\n\s*\n+", "\n", x)  # ยุบหลายบรรทัดให้เหลือบรรทัดเดียว
        x = "\n".join(line.strip() for line in x.splitlines()).strip()
    else:
        # ยุบทุกอย่างเป็นบรรทัดเดียว
        x = re.sub(r"\s+", " ", x).strip()

    if lower:
        x = x.lower()
    return x

In [7]:
# Storing in "sw_set" the set of English stopwords provided by nltk
# Defining and applying the function "sw_remove" which remove stopwords from reviews
# Storing in "after_removal" the example of review after removal of the stopwords

sw_set = set(thai_stopwords())

def tokenize(text):
    return word_tokenize(text, engine="newmm", keep_whitespace=False)
    
def sw_remove(x):
    words = tokenize(x)
    filtered_list = [word for word in words if word not in sw_set and word.strip() != ""]
    return filtered_list

# preprocessing
def preprocessing(dataset):
    dataset = dataset.apply(lambda x: process(sw_remove(x)))
    return dataset

# implement preprocessing

In [8]:
print("preprocessing...\n")
print("take long times...")
X_train = preprocessing(X_train)
X_val = preprocessing(X_val)
X_test = preprocessing(X_test)

print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_val:", X_val.shape, "y_val:", y_val.shape)
print("X_test:", X_test.shape, "y_test:", y_test.shape)

X_train: (400,) y_train: (400, 12)
X_val: (200,) y_val: (200, 12)
X_test: (200,) y_test: (200, 12)


#**Text to Vector by word2vec**

In [9]:
counter = Counter()
for t in X_train:
    counter.update(t)

vocab = {PAD_TOKEN: PAD_ID, UNK_TOKEN: UNK_ID}
for i, (w, _) in enumerate(counter.most_common(MAX_VOCAB - 2), start=2):
    vocab[w] = i

def encode(text):
    ids = [vocab.get(t, vocab[UNK_TOKEN]) for t in text][:MAX_LEN]
    if len(ids) < MAX_LEN:
       ids = ids + [PAD_ID] * (MAX_LEN - len(ids))
    return ids

vocab_size = len(vocab)

In [10]:
vectorize_model = word_vector.WordVector(model_name="thai2fit_wv").get_model() # load thai2fit_wv from pythainlp
# thai2vec_dim = vectorize_model.vector_size

Corpus: thai2fit_wv
- Downloading: thai2fit_wv 0.1


  0%|          | 0/62452646 [00:00<?, ?it/s]

# สร้าง embedding matrix ตาม vocab for initial weight

In [11]:
embedding_weight= np.zeros((vocab_size, HIDDEN_DIM), dtype=np.float32)

# ตารางคำศัพท์ → เวกเตอร์
# PAD row (0) = 0 
# UNK row (1) สุ่มเล็กน้อย
rng = np.random.default_rng(RANDOM_STATE)
embedding_weight[UNK_ID] = rng.normal(0, 0.01, size=(HIDDEN_DIM,)).astype(np.float32)

for word, idx in vocab.items():
    if word in (PAD_TOKEN, UNK_TOKEN):
        continue
    if word in vectorize_model:
        embedding_weight[idx] = vectorize_model[word]
print(embedding_weight.shape)

(100, 300)


# RNN Part

In [12]:
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [13]:
class PhachathaiDataset(Dataset):
    def __init__(self, X, y):
        self.X = X.tolist() # list of list[str]
        self.y = y.astype(np.int64).to_numpy()

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        ids = encode(self.X[idx])   # list[int] length MAX_LEN
        return torch.tensor(ids, dtype=torch.long), torch.tensor(self.y[idx], dtype=torch.float32)

In [14]:
class RNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, output_dim=2, embedding_matrix=None):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        if embedding_matrix is not None:
            self.embedding.weight.data.copy_(torch.tensor(embedding_matrix, dtype=torch.float32))
        self.embedding.weight.data[PAD_ID].zero_()
        self.embedding.weight.requires_grad = True
        
        self.rnn = nn.LSTM(embed_dim, embed_dim // 2, batch_first=True)
        self.fc1 = nn.Linear(embed_dim // 2, embed_dim // 4)
        self.relu = nn.ReLU()
        self.fc_dropout = nn.Dropout(0.3)
        self.fc2 = nn.Linear(embed_dim // 4, output_dim)

    def forward(self, text): # B, T
        lengths = (text != 0).sum(dim=1)
        embedded = self.embedding(text) # (B,T,D)
        packed = nn.utils.rnn.pack_padded_sequence(embedded, lengths.cpu(), enforce_sorted=False, batch_first=True)
        _, (hidden, _) = self.rnn(packed)
        out = hidden[-1, :, :]
        out = self.fc1(out)
        out = self.relu(out)
        out = self.fc_dropout(out)
        out = self.fc2(out)
        return out

In [15]:
model = RNN(vocab_size, embed_dim=HIDDEN_DIM, embedding_matrix=embedding_weight, output_dim=NUM_LABEL).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

 # pos_weight สำหรับ imbalance (กัน label ที่หายาก) 
pos = y_train.sum(axis=0)
neg = y_train.shape[0] - pos
pos_weight = torch.tensor(neg / (pos + 1e-8), dtype=torch.float32).to(DEVICE)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

/tmp/ipykernel_55/3011824564.py:7: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  pos_weight = torch.tensor(neg / (pos + 1e-8), dtype=torch.float32).to(DEVICE)


In [16]:
@torch.no_grad()
def evaluate(model, loader, criterion, device, threshold=0.5):
    model.eval()
    total_loss, total = 0.0, 0

    all_preds = []
    all_labels = []

    for x, y in loader:
        x, y = x.to(device), y.to(device)

        logits = model(x)
        loss = criterion(logits, y)
        probs = torch.sigmoid(logits)
        probs = (probs > threshold).int()

        total_loss += loss.item() * y.size(0)
        total += y.size(0)


        all_preds.append(probs.detach().cpu().numpy())
        all_labels.append(y.detach().cpu().numpy())

    y_pred = np.concatenate(all_preds)
    y_true = np.concatenate(all_labels)

    acc = (y_pred == y_true).mean()
    f1_micro = f1_score(y_true, y_pred, average="micro")  
    f1_macro = f1_score(y_true, y_pred, average="macro")

    return float(total_loss / total), acc, float(f1_micro), float(f1_macro)

In [17]:
def train_one_epoch(model, loader, optimizer, criterion, device, threshold):
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    for x, y in loader:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad(set_to_none=True)
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * y.size(0)
        total += y.size(0)

    return float(running_loss / total)

In [18]:
def fit(model, train_loader, val_loader, optimizer, criterion, device, epochs, threshold):
    history = {"train_loss": [], "val_loss": [], "val_acc": [], "val_f1_micro": [], "val_f1_macro": []}

    for epoch in range(1, epochs + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device, threshold)
        val_loss, val_acc, val_f1_micro, val_f1_macro = evaluate(model, val_loader, criterion, device, threshold)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        history["val_f1_micro"].append(val_f1_micro)
        history["val_f1_macro"].append(val_f1_macro)

        print(
            f"Epoch {epoch:02d} | "
            f"train_loss={train_loss:.4f} | "
            f"val_loss={val_loss:.4f} val_acc={val_acc*100:.2f}% | "
            f"F1-micro={val_f1_micro:.4f} F1-macro={val_f1_macro:.4f} "
        )

    return history

In [19]:
def test(model, test_loader, criterion, device):
    test_loss, test_acc, test_f1_micro, test_f1_macro = evaluate(model, test_loader, criterion, device)
    print(f"TEST | loss={test_loss:.4f} acc={test_acc*100:.2f}% f1-micro={test_f1_micro:.4f} f1-macro={test_f1_macro:.4f}")
    return test_loss, test_acc, test_f1_micro, test_f1_macro

In [20]:
train_ds =PhachathaiDataset(X_train, y_train)
val_ds   = PhachathaiDataset(X_val, y_val)
test_ds  = PhachathaiDataset(X_test, y_test)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

# Train and Test

In [21]:
history = fit(model, train_loader, val_loader, optimizer, criterion, DEVICE, epochs=EPOCHS, threshold=THRESHOLD)

Epoch 01 | train_loss=1.2047 | val_loss=1.2549 val_acc=44.54% | F1-micro=0.2221 F1-macro=0.1355 
Epoch 02 | train_loss=1.2006 | val_loss=1.2549 val_acc=44.71% | F1-micro=0.2378 F1-macro=0.1491 
Epoch 03 | train_loss=1.1967 | val_loss=1.2537 val_acc=46.83% | F1-micro=0.2503 F1-macro=0.1594 
Epoch 04 | train_loss=1.1941 | val_loss=1.2530 val_acc=49.04% | F1-micro=0.2592 F1-macro=0.1739 
Epoch 05 | train_loss=1.1899 | val_loss=1.2539 val_acc=50.92% | F1-micro=0.2525 F1-macro=0.1726 


In [22]:
test_loss, test_acc,test_f1_micro, test_f1_macro = test(model, test_loader, criterion, DEVICE)

TEST | loss=1.2485 acc=52.25% f1-micro=0.2625 f1-macro=0.1818


In [23]:
@torch.no_grad()
def predict_text(model, text, device):
    model.eval()

    # preprocess เหมือน train
    text = process(text)
    text = sw_remove(text)
    encode_text = encode(text)
    x = torch.tensor([encode_text], dtype=torch.long, device=device)
    logits = model(x)
    logits = torch.sigmoid(logits)
    probs = (logits >= THRESHOLD).int()

    return logits, probs.detach().cpu().numpy()

In [24]:
logits, probs = predict_text(model,"พรรคประชาชนเตรียมจัดงาน มหกรรมประชาชนอาเซียน พร้อมปลูกข้าวสาร", DEVICE)
print("result:",probs)
print("show probs every labels")
for i in range(len(LABEL_COLS)):
    print(LABEL_COLS[i], ":", probs[0][i])

result: [[1 0 1 0 0 0 1 0 1 1 1 1]]
resul_prob: tensor([[0.5256, 0.4996, 0.5025, 0.4907, 0.4788, 0.4774, 0.5276, 0.4750, 0.5065,
         0.5152, 0.5111, 0.5154]], device='cuda:0')
